In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
from scipy.stats import norm
from hmmlearn.hmm import GaussianHMM
from arch import arch_model

# Path to the clean data we just made
potential_paths = [
    "../data/processed/eurusd_clean_returns_jan2026.parquet",
    "code/data/processed/eurusd_clean_returns_jan2026.parquet",
    "./data/processed/eurusd_clean_returns_jan2026.parquet",
    "data/processed/eurusd_clean_returns_jan2026.parquet"
]
data_path = next((p for p in potential_paths if os.path.exists(p)), potential_paths[0])
model_dir = "../models/" if os.path.exists("../models") else "code/models/"
os.makedirs(model_dir, exist_ok=True)

Current Working Directory: /content
❌ ERROR: Could not find 'eurusd_clean_returns_jan2026.parquet'.
Make sure you ran 'real_prep.ipynb' successfully first!


In [14]:
import os
print(os.listdir("/content/stk-mat2011/code/data/processed/"))


['README.md']


In [13]:
df = pd.read_parquet(data_path)
y = df['returns'].values
T = len(y)
ALPHA = 0.05
z = norm.ppf(1 - ALPHA / 2)

print(f"Loaded {T} observations. Sample Std: {np.std(y):.6f}")

TypeError: Expected a path-like, list of path-likes or a list of Datasets instead of the given type: NoneType

In [ ]:
""" Calculate Baselines on Real Data """
std_global = np.std(y)
mu_global = np.mean(y)

# Baseline: Always Mean
l0, u0 = mu_global - z*std_global, mu_global + z*std_global

# Baseline: Repeat Last (Momentum)
l_rep, u_rep = y[:-1] - z*std_global, y[:-1] + z*std_global
y_true = y[1:] # Align for scoring

print("Baselines ready.")

In [ ]:
""" HMM One-Step Prediction Intervals """
# predict_proba returns smoothed P(S_t | observations_1:T)
# This uses future info. We project forward using transmat for real prediction.
smoothed_probs = hmm_model.predict_proba(X) 
transmat = hmm_model.transmat_

# Project filtered/smoothed state to next step
pred_probs = np.dot(smoothed_probs[:-1], transmat)

mus = hmm_model.means_.flatten()
sigs = np.sqrt(hmm_model.covars_.flatten())

# Weighted prediction mean and variance
y_hat_hmm = np.sum(pred_probs * mus, axis=1)
y_var_hmm = np.sum(pred_probs * (sigs**2 + (mus - y_hat_hmm.reshape(-1,1))**2), axis=1)
y_std_hmm = np.sqrt(y_var_hmm)

l_hmm, u_hmm = y_hat_hmm - z*y_std_hmm, y_hat_hmm + z*y_std_hmm
print(f"HMM Intervals generated for {len(l_hmm)} steps.")

In [ ]:
""" HMM Mixed Prediction Intervals """
probs = hmm_model.predict_proba(X) # P(S_t | observations)

# Filter parameters
mus = hmm_model.means_.flatten()
sigs = np.sqrt(hmm_model.covars_.flatten())

# Weighted prediction mean and variance
# We use probs[t-1] to predict y[t]
y_hat_hmm = np.sum(probs[:-1] * mus, axis=1)
y_var_hmm = np.sum(probs[:-1] * (sigs**2 + (mus - y_hat_hmm.reshape(-1,1))**2), axis=1)
y_std_hmm = np.sqrt(y_var_hmm)

l_hmm, u_hmm = y_hat_hmm - z*y_std_hmm, y_hat_hmm + z*y_std_hmm
print("HMM Intervals generated.")

In [ ]:
""" Final Results on Real EUR/USD Data """

def score(y, l, u):
    # Broadcast scalars if needed
    l = np.full_like(y, l) if np.isscalar(l) else l
    u = np.full_like(y, u) if np.isscalar(u) else u
    
    cov = np.mean((y >= l) & (y <= u))
    wid = np.mean(u - l)
    is_v = np.mean((u - l) + (2/ALPHA)*np.maximum(l-y, 0) + (2/ALPHA)*np.maximum(y-u, 0))
    return cov, wid, is_v

results = {
    "Baseline: Global Std": score(y_true, l0, u0),
    "Baseline: Repeat Last": score(y_true, l_rep, u_rep),
    "HMM (One-Step Pred)": score(y_true, l_hmm, u_hmm),
    "GARCH(1,1)": score(y_true, l_g, u_g)
}

print("=" * 80)
print("REAL DATA ANALYSIS: EUR/USD JAN 2026 ".center(80))
print("=" * 80)
print(f"{'Model':<25} | {'Coverage':>8} | {'Avg Width':>10} | {'IS (↓)':>10}")
print("-" * 80)
for name, m in results.items():
    print(f"{name:<25} | {m[0]:>8.4f} | {m[1]:>10.4f} | {m[2]:>10.4f}")
print("=" * 80)

# Visualization
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title("Volatility States")
s = hmm_model.predict(X)
plt.plot(np.cumsum(y), color='silver', alpha=0.5, label='Cum. Returns')
for i in range(len(mus)):
    mask = s == i
    plt.scatter(np.where(mask)[0], np.cumsum(y)[mask], s=2, label=f'State {i}')
plt.legend()

plt.subplot(1, 2, 2)
plt.title("Interval Performance (Zoomed)")
chunk = range(min(100, len(y_true)), min(250, len(y_true)))
if len(chunk) > 0:
    plt.plot(y_true[chunk], color='black', alpha=0.2, label='Returns')
    plt.fill_between(chunk, l_hmm[chunk], u_hmm[chunk], color='blue', alpha=0.3, label='HMM')
    plt.fill_between(chunk, l_g[chunk], u_g[chunk], color='red', alpha=0.2, label='GARCH')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
""" Final Results on Real EUR/USD Data """

def score(y, l, u):
    cov = np.mean((y >= l) & (y <= u))
    wid = np.mean(u - l)
    is_v = np.mean((u - l) + (2/ALPHA)*np.maximum(l-y, 0) + (2/ALPHA)*np.maximum(y-u, 0))
    return cov, wid, is_v

results = {
    "Baseline: Global Std": score(y_true, l0, u0),
    "Baseline: Repeat Last": score(y_true, l_rep, u_rep),
    "HMM (Inferred State)": score(y_true, l_hmm, u_hmm),
    "GARCH(1,1)": score(y_true, l_g, u_g)
}

print("=" * 80)
print("REAL DATA ANALYSIS: EUR/USD JAN 2026".center(80))
print("=" * 80)
print(f"{'Model':<25} | {'Coverage':>8} | {'Avg Width':>10} | {'IS (↓)':>10}")
print("-" * 80)
for name, m in results.items():
    print(f"{name:<25} | {m[0]:>8.4f} | {m[1]:>10.4f} | {m[2]:>10.4f}")
print("=" * 80)